# C3 Water Security Digital Twin: Household Water-Demand Forecasting
### Google Colab Training & Evaluation Pipeline

Source: AI/C3/data/raw/DataSet.xlsx, Consumption_Data and Household_Info worksheets. Consumption units are unspecified. Review C3_Consumption_Validation.md and confirm units and meter anomalies before training. No model training is performed locally. Training examples below remain commented out.

**Workflow Stages:**
1. **Load Data**: C3_Household_Consumption_Cleaned.csv contains all 1,782 actual monthly records for 99 households, with workbook household attributes included. No separate master CSV is required.
2. **Data Inspection**: Check date ranges, record counts per household, missing values, and distributions.
3. **Feature Engineering**: Generate temporal lags (`lag_1`, `lag_2`, `lag_3`), rolling statistics, and calendar features.
4. **Time-Based Split**: Chronological split. Shifted features describe rolling one-month-ahead evaluation using past actuals. Fixed-origin multi-month forecasting needs a separate evaluation protocol.
5. **Baseline Evaluation**: Evaluate Naive Persistence ($y_t = y_{t-1}$) and 3-period Moving Average.
6. **ML Model Training**: Train XGBoost and LightGBM Regressors.
7. **Benchmarking**: Compare MAE, RMSE, MAPE, and $R^2$.
8. **Export**: Save trained model artifact (`demand_forecast_model.pkl`) and forecast outputs for frontend dashboard integration.

In [ ]:
# 1. Install & Import Required Libraries
!pip install -q xgboost lightgbm scikit-learn pandas numpy matplotlib seaborn joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Environment initialized successfully.")

In [ ]:
# 2. Upload Cleaned Datasets
# Upload the actual export from AI/C3/data/processed/
from google.colab import files
print("Please upload C3_Household_Consumption_Cleaned.csv:")
uploaded = files.upload()

# Alternatively, mount Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# 3. Load & Inspect Actual Historical Consumption
df_consumption = pd.read_csv('C3_Household_Consumption_Cleaned.csv')
print('Households:', df_consumption.HouseholdID.nunique())
print('Records:', len(df_consumption))
print('Date range:', df_consumption.Date.min(), df_consumption.Date.max())
print(df_consumption.groupby('HouseholdID').size())
print('Missing:', df_consumption.isna().sum())
print('Duplicates:', df_consumption.duplicated(['HouseholdID', 'Date']).sum())
df_consumption.head()

In [ ]:
# 4. Preprocessing & Temporal Feature Engineering
# Consumption stays in unspecified source units; no conversion or imputation.
def prepare_features(consumption_file):
    df = pd.read_csv(consumption_file)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(by=['HouseholdID', 'Date']).reset_index(drop=True)
    
    # Household attributes already come from the actual workbook.
    required = ['HouseholdID', 'Date', 'Consumption', 'Zone', 'DMA', 'TankCapacityLitres']
    if df[required].isna().any().any():
        raise ValueError('Missing required values')
    if df.duplicated(['HouseholdID', 'Date']).any():
        raise ValueError('Duplicate household/date records')
    df['Consumption'] = pd.to_numeric(df['Consumption'], errors='raise')
    if (df['Consumption'] < 0).any():
        raise ValueError('Negative consumption requires review')
    
    # Temporal calendar features
    df['Month'] = df['Date'].dt.month
    df['Quarter'] = df['Date'].dt.quarter
    
    # Lags within each household
    grouped = df.groupby('HouseholdID')['Consumption']
    df['lag_1'] = grouped.shift(1)
    df['lag_2'] = grouped.shift(2)
    df['lag_3'] = grouped.shift(3)
    df['rolling_mean_3'] = grouped.transform(lambda s: s.shift(1).rolling(window=3).mean())
    df['rolling_std_3'] = grouped.transform(lambda s: s.shift(1).rolling(window=3).std())
    
    # Encode Zone
    df['Zone_Code'] = (df['Zone'] == 'Zone B').astype(int)
    
    return df

df_featured = prepare_features('C3_Household_Consumption_Cleaned.csv')

In [ ]:
# 5. Time-Based Train/Test Split (Strict Chronological Ordering)
def time_split(df, split_ratio=0.8):
    dates = df['Date'].drop_duplicates().sort_values().values
    split_idx = int(len(dates) * split_ratio)
    split_date = dates[split_idx]
    
    df_clean = df.dropna(subset=['lag_1', 'lag_2', 'lag_3', 'rolling_mean_3']).copy()
    train = df_clean[df_clean['Date'] < split_date]
    test = df_clean[df_clean['Date'] >= split_date]
    
    print(f"Split Date: {split_date}")
    print(f"Training records: {len(train)} | Testing records: {len(test)}")
    return train, test

features = ['lag_1', 'lag_2', 'lag_3', 'rolling_mean_3', 'Month', 'Quarter', 'TankCapacityLitres', 'Zone_Code']
target = 'Consumption'

In [ ]:
# 6. Baseline Models vs. Machine Learning Training
# Baseline 1: Naive Persistence (y_pred = lag_1)
# Baseline 2: 3-Period Moving Average (y_pred = rolling_mean_3)

def evaluate_predictions(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if np.any(mask) else np.nan
    r2 = r2_score(y_true, y_pred)
    return {
        'Model': model_name,
        'MAE (source units)': round(mae, 2),
        'RMSE (source units)': round(rmse, 2),
        'MAPE (%)': round(mape, 2),
        'R2 Score': round(r2, 4)
    }

In [ ]:
# 7. Train & Evaluate XGBoost & LightGBM
# Once train and test datasets are prepared:
# X_train, y_train = train[features], train[target]
# X_test, y_test = test[features], test[target]

# b1_metrics = evaluate_predictions(y_test, test['lag_1'], 'Baseline 1: Persistence')
# b2_metrics = evaluate_predictions(y_test, test['rolling_mean_3'], 'Baseline 2: 3-Month MA')

# xgb = XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=42)
# xgb.fit(X_train, y_train)
# xgb_pred = xgb.predict(X_test)
# xgb_metrics = evaluate_predictions(y_test, xgb_pred, 'XGBoost Regressor')

# comparison_df = pd.DataFrame([b1_metrics, b2_metrics, xgb_metrics])
# print(comparison_df.to_markdown(index=False))

In [ ]:
# 8. Save Trained Model & Download Artifacts
# joblib.dump(xgb, 'demand_forecast_model.pkl')
# files.download('demand_forecast_model.pkl')
print("Training & evaluation pipeline ready for execution in Google Colab.")